Checking PSD of R for DepthAnything v2 on COCO dataset

PSD of R <=> eigenvalues of R >= 0

In [2]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import numpy as np
import cv2

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git

Cloning into 'Depth-Anything-V2'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 146 (delta 32), reused 24 (delta 24), pack-reused 82 (from 1)
Receiving objects: 100% (146/146), 45.17 MiB | 14.06 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [5]:
import sys
sys.path.append("Depth-Anything-V2")

In [6]:
!ls

Depth-Anything-V2  sample_data


In [7]:
import sys
import os

# Ensure the parent directory of 'depth_anything_v2' is in sys.path
# The 'Depth-Anything-V2' repository is expected to be in '/content/'
repo_root_path = os.path.abspath(os.path.join(os.getcwd(), "Depth-Anything-V2"))
if repo_root_path not in sys.path:
    sys.path.insert(0, repo_root_path)

from depth_anything_v2.dpt import DepthAnythingV2

In [8]:
model = DepthAnythingV2("vits").to(device)
model.eval()

DepthAnythingV2(
  (pretrained): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-11): 12 x NestedTensorBlock(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
    

In [30]:
!wget http://images.cocodataset.org/zips/test2017.zip

--2026-03-28 14:07:52--  http://images.cocodataset.org/zips/test2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.184.171, 3.5.24.245, 16.15.191.2, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.184.171|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6646970404 (6.2G) [application/zip]
Saving to: ‘test2017.zip’

test2017.zip        100%[===================>]   6.19G  17.7MB/s    in 9m 57s  

2026-03-28 14:17:49 (10.6 MB/s) - ‘test2017.zip’ saved [6646970404/6646970404]



In [12]:
!unzip -q test2017.zip

In [13]:
test_dir = "test2017"
files = os.listdir(test_dir)
len(files)

40670

In [14]:
from torchvision.transforms import Compose

class NormalizeImage(object):

    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, sample):
        sample["image"] = (sample["image"] - self.mean) / self.std
        return sample

class Resize(object):

    def __init__(self, width, height, multiple_of=14):
        self.width, self.height = width, height
        self.multiple_of = multiple_of

    def constrain_to_multiple_of(self, x, min_val=0, max_val=None):
        y = (np.round(x / self.multiple_of) * self.multiple_of).astype(int)
        if max_val is not None and y > max_val:
            y = (np.floor(x / self.multiple_of) * self.multiple_of).astype(int)
        if y < min_val:
            y = (np.ceil(x / self.multiple_of) * self.multiple_of).astype(int)
        return y

    def get_size(self, width, height):
        # determine new height and width
        scale_height, scale_width = self.height / height, self.width / width
        # scale such that output size is lower bound
        if scale_width > scale_height:
            scale_height = scale_width # fit width
        else:
            scale_width = scale_height # fit height
        new_height = self.constrain_to_multiple_of(scale_height * height, min_val=self.height)
        new_width = self.constrain_to_multiple_of(scale_width * width, min_val=self.width)
        return (new_width, new_height)

    def __call__(self, sample):
        width, height = self.get_size(sample["image"].shape[1], sample["image"].shape[0])
        sample["image"] = cv2.resize(sample["image"], (width, height), interpolation=cv2.INTER_CUBIC) # resize sample
        return sample

class PrepareForNet(object):

    def __init__(self):
        pass

    def __call__(self, sample):
        image = np.transpose(sample["image"], (2, 0, 1))
        sample["image"] = np.ascontiguousarray(image).astype(np.float32)
        return sample

def preprocessing(raw_image, input_size=518):
    transform = Compose([
        Resize(width=input_size, height=input_size, multiple_of=14),
        NormalizeImage(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        PrepareForNet(),
    ])
    h, w = raw_image.shape[:2]
    image = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB) / 255.0
    image = transform({'image': image})['image']
    image = torch.from_numpy(image).unsqueeze(0)
    image = image.to(device)
    return image

In [15]:
model.pretrained

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (n

In [16]:
result = model.pretrained.get_intermediate_layers(preprocessing(cv2.imread(os.path.join(test_dir, files[0]))).to(device))

In [17]:
result[0].shape

torch.Size([1, 2035, 384])

In [20]:
from torch import Tensor
from typing import Literal

In [21]:
format2dim = {
    "hwc": 2,
    "chw": 0,
    "nc": 1,
    "bhwc": 3,
    "bchw": 1,
    "bnc": 2,
}

In [22]:
def fixingValue(
    features: Tensor, # (B,C,H,W), (B,H,W,C), (C,H,W), (H,W,C), (B,H*W,C), or (H*W,C) accordig to `data_format`
    data_format: Literal["hwc", "chw", "nc", "bhwc", "bchw", "bnc" ] = "chw",
    eps: float = 1e-5,
):
    dim = format2dim[data_format]
    value = features.norm(dim=dim).abs().max() + eps
    return value.max().item()

In [24]:
def extendWithFix(
    features: Tensor, # (B,C,H,W), (B,H,W,C), (C,H,W), (H,W,C), (B,H*W,C), or (H*W,C) accordig to `data_format`
    data_format: Literal["hwc", "chw", "nc", "bhwc", "bchw", "bnc" ] = "chw",
    eps: float = 1e-5,
) -> Tensor:
    """Extend the cosine similarity features by adding a feature that ensures the ncut algorithm is applicable."""
    dim = format2dim[data_format]
    value = fixingValue(features, data_format, eps)
    shape = list(features.shape)
    shape[dim] = 1
    shape = tuple(shape)
    extra = torch.full(shape, value, device=features.device, dtype=features.dtype)
    return torch.cat([features, extra], dim=dim) # (B,C+1,H,W), (B,H,W,C+1), (B,N,C+1), (C+1,H,W), (H,W,C+1), or (N,C+1) accordig to `data_format`

In [25]:
img = cv2.imread(os.path.join(test_dir, files[0]))
blob = preprocessing(img).to(device)
with torch.no_grad():
    result = model.pretrained.get_intermediate_layers(blob,[11])
    feats = result[0][0][1:]
    W = torch.matmul(feats, feats.T)
    print(W.min().item(),W.max().item())
    d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
    print(d.min().item(),d.max().item())

-338.5190124511719 384.0001220703125
-51024.515625 190395.0625


In [29]:
feats.shape

torch.Size([2034, 385])

In [32]:
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    blob = preprocessing(img).to(device)
    with torch.no_grad():
        result = model.pretrained.get_intermediate_layers(blob,[11])
        feats = result[0][0][1:]
    W = torch.matmul(feats, feats.T)
    minW = W.min().item()
    if minW <= 0:
        print(i,'W',minW,"!!!!!" if minW <= 0 else "")
    d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
    mind = d.min().item()
    if mind <= 0:
        print(i,'d',mind,"!!!!!" if mind <= 0 else "")
    else:
        D = torch.diag(d.squeeze(-1))
        eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)
        if eigenvalues[0] < -1e5:
            print(i,'eig',eigenvalues[0].item(),"!!!!!" if eigenvalues[0].item() <= 0 else "")
    if i == 100:
        break

0 W -338.5190124511719 !!!!!
0 d -51024.515625 !!!!!
1 W -213.069091796875 !!!!!
1 d -154495.640625 !!!!!
2 W -332.021728515625 !!!!!
2 d -13937.91015625 !!!!!
3 W -320.35455322265625 !!!!!
3 d -15027.96875 !!!!!
4 W -324.5871276855469 !!!!!
4 d -19380.380859375 !!!!!
5 W -326.0675354003906 !!!!!
5 d -164883.140625 !!!!!
6 W -316.0289611816406 !!!!!
7 W -302.50335693359375 !!!!!
7 d -138986.046875 !!!!!
8 W -301.6304016113281 !!!!!
8 d -142077.59375 !!!!!
9 W -337.2825622558594 !!!!!
9 d -49337.9140625 !!!!!
10 W -330.4732360839844 !!!!!
10 d -42825.90625 !!!!!
11 W -262.7423400878906 !!!!!
11 d -184130.921875 !!!!!
12 W -319.0743713378906 !!!!!
12 d -141808.53125 !!!!!
13 W -255.5159149169922 !!!!!
14 W -224.87074279785156 !!!!!
14 d -99048.3046875 !!!!!
15 W -252.8619384765625 !!!!!
15 d -59405.328125 !!!!!
16 W -341.427490234375 !!!!!
17 W -206.1387481689453 !!!!!
17 d -9429.724609375 !!!!!
18 W -305.03076171875 !!!!!
18 d -68035.046875 !!!!!
19 W -268.09210205078125 !!!!!
19 d -186

The values of W might be negative, d might not be well-defined, and R might not be PSD.

In [28]:
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    blob = preprocessing(img).to(device)
    with torch.no_grad():
        result = model.pretrained.get_intermediate_layers(blob,[11])
        feats0 = result[0][0][1:]
        feats = extendWithFix(feats0,data_format="nc")
    W = torch.matmul(feats, feats.T)
    minW = W.min().item()
    if minW <= 0:
        print(i,minW,"!!!!!" if minW <= 0 else "")
    d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
    mind = d.min().item()
    if mind <= 0:
        print(i,mind,"!!!!!" if mind <= 0 else "")
    D = torch.diag(d.squeeze(-1))
    eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)
    if eigenvalues[0] < -1e5:
        print(i,eigenvalues[0],"!!!!!" if eigenvalues[0] <= 0 else "")
        break
    if i % 500 == 0:
        print('done',i,'/',len(files),minW,mind,eigenvalues[0].item(),eigenvalues[1].item())

done 0 / 40670 45.481239318847656 730032.0 7.106992683247881e-08 0.611382007598877
done 500 / 40670 66.28824615478516 614469.6875 -1.5810896059065271e-07 0.5687914490699768
done 1000 / 40670 220.4368896484375 691201.4375 2.7970372684649192e-08 0.9369430541992188
done 1500 / 40670 75.4626235961914 757590.3125 2.0843324932684482e-07 0.8036434650421143
done 2000 / 40670 85.46825408935547 634205.625 -3.442915996743068e-08 0.5164870619773865
done 2500 / 40670 66.82596588134766 613170.5625 1.4894465039105853e-07 0.6011384129524231
done 3000 / 40670 89.05126190185547 762511.25 -9.265266953661921e-08 0.6352975368499756
done 3500 / 40670 66.12960815429688 453742.5 -2.2922310449757788e-07 0.6851441860198975
done 4000 / 40670 77.41019439697266 771825.25 -6.9188487827887e-08 0.7706084847450256
done 4500 / 40670 49.67595672607422 710694.1875 1.6270918479222019e-07 0.4976474940776825
done 5000 / 40670 93.28215789794922 415924.84375 -3.024675265805854e-08 0.6889017224311829
done 5500 / 40670 167.2669

However with the fix, the values of W are positive and d is well defined, and R is always PSD